<a href="https://colab.research.google.com/github/daniakhan123/flyrank-ml-internship-starter/blob/main/w03_data_contract_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniakhan123/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**

## 0. Setup — connect to the warehouse

*Run this once. Put your Hugging Face **READ** token in Colab's Secrets panel (key icon, left sidebar) as `HF_TOKEN` — never paste it into a cell, this repo is public.*

In [ ]:
%pip -q install duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

# --- CONFIG: the one place to fix things if a column name below doesn't match your repo's
# docs/data-dictionary.md when you run the DESCRIBE cell just below. Everything downstream
# reads from this dict, so a mismatch is a one-line fix, not a full rewrite. ---
MONTH_DEV   = "2026-03"   # mid-panel month we iterate on (never the sealed final month)
MONTH_NEXT  = "2026-04"   # the month right after, used only to build the forward-looking label
COLS = {
    "date":            "report_date",
    "client_key":      "client_hash_id",
    "content_key":     "content_hash_id",
    "impressions":      "gsc_impressions",
    "clicks":           "gsc_clicks",
    "position":         "gsc_avg_position",
    "gsc_available":    "gsc_data_available",
    "ga4_available":    "ga4_data_available",
    "sessions":         "sessions_ga4",
    "sessions_ai":      "sessions_ai",
}

FACT = f"read_parquet('{WAREHOUSE}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{WAREHOUSE}/dim_content/**/*.parquet')"
DIM_CLIENTS = f"read_parquet('{WAREHOUSE}/dim_clients/**/*.parquet')"
print('Connected. Dev month:', MONTH_DEV, '| Next month (label only):', MONTH_NEXT)

Connected. Dev month: 2026-03 | Next month (label only): 2026-04


In [ ]:

schema = con.sql(f"DESCRIBE SELECT * FROM {FACT} WHERE month = '{MONTH_DEV}' LIMIT 1").df()
print(schema[['column_name', 'column_type']].to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row** in `fact_content_daily_performance` = one content item, for one client, on one calendar day — a single day's Google Search Console + GA4 snapshot for that page (grain confirmed in `docs/ml-intern-dataset-and-lane-guide.md`: `report_date + client_hash_id + content_hash_id`).

**Tables used:** `fact_content_daily_performance` (the daily fact, partitioned by `month=YYYY-MM`), joined to `dim_content` for content metadata (`content_hash_id`) and to `dim_clients` for per-client history coverage (`client_hash_id`, `gsc_data_start`, `ga4_data_start`).

**Time window:** I iterate on **`month=2026-03`** — a mid-panel month, away from both the panel's ragged start and the sealed final month. The forward-looking label (Section 3) additionally reads **`month=2026-04`**, the very next month, only to look up what happened *after* the March decision point — never to build a feature. `fact_content_daily_performance_sample` (June 2026, the final month) is treated as a **sealed test month** and is not touched anywhere in this notebook.

In [ ]:
# text answer is in the markdown cell above — this cell intentionally left for Run All
pass

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**What I'd predict / rank (label or proxy):** whether a page's search visibility is about to **decline** — a forward-looking proxy, `will_decline_next_month`, built the same way the starter CSV's `is_declining_label` is built (never from a rule FlyRank's product already computed): 1 when total GSC clicks in `month=2026-04` are lower than in `month=2026-03` for that `(client_hash_id, content_hash_id)` pair, else 0. This is what a refresh reviewer would want ranked: pages about to lose ground get looked at first.

**One thing I deliberately exclude:** `sessions_ai` (AI-referral sessions). The lane guide's density table shows only 30,177 rows out of 78.8M have any AI-session data — far too sparse for this month's slice to carry real signal, and treating its near-universal zero as "no AI traffic" would be noise dressed up as a feature. I leave it out of this lane and would revisit it only with a purpose-built AI-referral slice.

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (March only), `sessions_ga4` (March only, `ga4_data_available IS TRUE`), `days_with_impressions` (derived) | All measured *within* the March decision window — known before the April outcome |
| **Label / proxy** | `will_decline_next_month` (built from April clicks vs. March clicks) | The target — never a feature |
| **Context** | `report_date`, `client_hash_id`, `content_hash_id`, `month` | Joining, grouping, and the client-holdout split only — never fed to a model |
| **Excluded** | `sessions_ai` (sparsity, see above); any `health_score` / `priority_score` / `action_type` style field — not shipped in this data anyway, and would be a product decision, not an observed signal |


In [ ]:
# text answer is in the markdown cell above — this cell intentionally left for Run All
pass

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain: one row really is one (day, client, content)

In [ ]:
grain_check = con.sql(f"""
    SELECT {COLS['date']}, {COLS['client_key']}, {COLS['content_key']}, COUNT(*) AS c
    FROM {FACT}
    WHERE month = '{MONTH_DEV}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f'Duplicate-grain rows found: {len(grain_check)} (0 means the stated grain holds)')
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate-grain rows found: 0 (0 means the stated grain holds)


,report_date,client_hash_id,content_hash_id,c


### Query 2 — Slice size: row count and date span for `month=2026-03`

In [ ]:
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN({COLS['date']}) AS min_date,
           MAX({COLS['date']}) AS max_date,
           COUNT(DISTINCT {COLS['client_key']}) AS n_clients,
           COUNT(DISTINCT {COLS['content_key']}) AS n_content_items
    FROM {FACT}
    WHERE month = '{MONTH_DEV}'
""").df()
slice_stats

,n_rows,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


### Query 3 — Availability: filter with `IS TRUE`, show how many rows survive

The panel is unbalanced (`docs/ml-intern-dataset-and-lane-guide.md`, §2) — rows before a client's `ga4_data_start` have the GA4 columns zero-filled, flagged with `ga4_data_available`, and that flag can be **NULL**, not just TRUE/FALSE. `= FALSE` or `NOT flag` silently drops the NULLs into the wrong bucket — only `IS TRUE` is safe.

In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE {COLS['ga4_available']} IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE {COLS['ga4_available']} IS FALSE) AS ga4_unavailable_rows,
        COUNT(*) FILTER (WHERE {COLS['ga4_available']} IS NULL)  AS ga4_null_rows
    FROM {FACT}
    WHERE month = '{MONTH_DEV}'
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_unavailable_rows,ga4_null_rows
0,9841378,413966,6408671,3018741


### Five features (`month=2026-03`)

One row per `(client_hash_id, content_hash_id)`, aggregated over March only — every value below is knowable at the decision point (end of March), because nothing in it reaches past `month=2026-03`.

In [ ]:
features = con.sql(f"""
    SELECT
        {COLS['client_key']}  AS client_key,
        {COLS['content_key']} AS content_key,
        SUM({COLS['impressions']}) AS impressions_month,
        SUM({COLS['clicks']}) AS clicks_month,
        AVG({COLS['position']}) AS avg_position_month,
        SUM(ga4_sessions) FILTER (
            WHERE {COLS['ga4_available']} IS TRUE
        ) AS sessions_month,
        COUNT(*) FILTER (
            WHERE {COLS['impressions']} > 0
        ) AS days_with_impressions
    FROM {FACT}
    WHERE month = '{MONTH_DEV}'
    GROUP BY 1, 2
""").df()

print('feature frame shape:', features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature frame shape: (331437, 7)


,client_key,content_key,impressions_month,clicks_month,avg_position_month,sessions_month,days_with_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,NaN,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,31
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,21


1. **`impressions_month`** — knowable at the decision moment because it's the sum of a GSC metric observed *during* March itself, nothing from April is touched.
2. **`clicks_month`** — same reasoning: a March-only sum, observed before the end-of-March review point.
3. **`avg_position_month`** — a March-only average of the daily GSC position; the ranking that already happened, not one that hasn't happened yet.
4. **`sessions_month`** — March-only GA4 sessions, filtered to `ga4_data_available IS TRUE` so rows before that client's GA4 start (zero-filled, not "zero traffic") don't get counted as real zeros.
5. **`days_with_impressions`** — a March-only count of active days; it describes consistency within the window that's already closed, so it's known before anyone decides anything.

### The trap — add one label-derived column on purpose

Build the forward label from April, join it on, then (on purpose) add a feature that is really just the label in disguise — the exact leakage notebook 02 warned about, now on real warehouse data.

In [ ]:
next_month = con.sql(f"""
    SELECT
        {COLS['client_key']}  AS client_key,
        {COLS['content_key']} AS content_key,
        SUM({COLS['clicks']}) AS clicks_next_month
    FROM {FACT}
    WHERE month = '{MONTH_NEXT}'
    GROUP BY 1, 2
""").df()

labeled = features.merge(next_month, on=['client_key', 'content_key'], how='inner')
labeled['will_decline_next_month'] = (labeled['clicks_next_month'] < labeled['clicks_month']).astype(int)
print('rows with both months present:', len(labeled))
print('decline rate:', labeled['will_decline_next_month'].mean().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows with both months present: 331436
decline rate: 0.136


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

honest_features = ['impressions_month', 'clicks_month', 'avg_position_month',
                    'sessions_month', 'days_with_impressions']
X = labeled[honest_features].fillna(0)
y = labeled['will_decline_next_month']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
print(f'HONEST score (5 real features only), ROC-AUC: {honest_auc:.3f}')

HONEST score (5 real features only), ROC-AUC: 0.966


In [ ]:

labeled['clicks_pct_change'] = (
    (labeled['clicks_next_month'] - labeled['clicks_month'])
    / labeled['clicks_month'].replace(0, 1)
)

leaky_features = honest_features + ['clicks_pct_change']
X_leaky = labeled[leaky_features].fillna(0)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42, stratify=y
)
clf_leaky = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leaky.fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, clf_leaky.predict_proba(X_test_l)[:, 1])
print(f'LEAKY score (with clicks_pct_change added), ROC-AUC: {leaky_auc:.3f}')
print(f'Jump: {honest_auc:.3f} -> {leaky_auc:.3f}')

LEAKY score (with clicks_pct_change added), ROC-AUC: 1.000
Jump: 0.966 -> 1.000


In [ ]:
# Delete the leaked column and keep the honest number.
del labeled['clicks_pct_change']
print('clicks_pct_change removed.')
print(f'Kept, honest score: ROC-AUC = {honest_auc:.3f} (from the 5-feature frame only)')

clicks_pct_change removed.
Kept, honest score: ROC-AUC = 0.966 (from the 5-feature frame only)


**Why the jump happened:** `clicks_pct_change` is arithmetically derived from `clicks_next_month` — the same number the label `will_decline_next_month` is thresholded on. The "feature" isn't predicting the future, it's restating it. That's why the score jumped toward-perfect the instant it was added, and why it's gone for good, not just hidden — the honest score above is the one that counts.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** this slice cannot tell whether a **refresh would actually work** — it can only rank pages by observed decline risk. `fact_content_daily_performance` has no record of whether a page was ever refreshed, so there is no way, from this data alone, to measure a refresh's causal effect on recovery; that would need an explicit before/after experiment design, not a historical query (`docs/ml-intern-dataset-and-lane-guide.md`, §7). A secondary limit worth naming: because the panel is unbalanced (`dim_clients.gsc_data_start` / `ga4_data_start` differ per client), the March slice over-represents clients with longer tracking history — a single global calendar window is not the same as a fair per-client window.